### **Label Encoder**

#### It is used to convert string categories into integers

In [2]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

data = ['cat', 'dog', 'fish', 'dog', 'cat', 'fish', 'cat']

encoded = le.fit_transform(data)
print(encoded)
print(le.classes_)

[0 1 2 1 0 2 0]
['cat' 'dog' 'fish']


One important thing to note:
LabelEncoder assigned 0, 1, 2 — but this introduces a false ordering. The model might interpret fish(2) > dog(1) > cat(0) which is meaningless for animal categories.
This is why LabelEncoder should only be used for the target variable y — not for features. For features, use OrdinalEncoder instead.

In [3]:
# converting number back to original labels

print(le.inverse_transform([2,0,1]))

['fish' 'cat' 'dog']


## Ordinal Encoder

OrdinalEncoder is used for **features** (unlike LabelEncoder which is for targets).
It converts categorical string values into integers, but unlike LabelEncoder:
- It works on **2D input** (one or more columns at a time)
- You can **define a custom order** for the categories

Use OrdinalEncoder when categories have a **natural ranking** — like education levels, satisfaction ratings, or size (Small < Medium < Large).

**Key rule:** Never use LabelEncoder on features — use OrdinalEncoder instead.

Let's see this with the Adult Income dataset.

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

df = pd.read_csv('../datasets/adult.csv')
df = df.replace('?', np.nan)
df = df.dropna(subset=['workclass', 'occupation'])
df['native-country'] = df['native-country'].fillna('Unknown')
df = df.drop(columns=['fnlwgt', 'educational-num'])

In [8]:
print(df['education'].unique())


<StringArray>
[        '11th',      'HS-grad',   'Assoc-acdm', 'Some-college',
         '10th',  'Prof-school',      '7th-8th',    'Bachelors',
      'Masters',    'Doctorate',      '5th-6th',    'Assoc-voc',
          '9th',         '12th',      '1st-4th',    'Preschool']
Length: 16, dtype: str


In [9]:
education_order = [
    'Preschool', '1st-4th', '5th-6th', '7th-8th', '9th',
    '10th', '11th', '12th', 'HS-grad', 'Some-college',
    'Assoc-voc', 'Assoc-acdm', 'Bachelors', 'Masters',
    'Prof-school', 'Doctorate'
]

oe = OrdinalEncoder(categories=[education_order])

# OrdinalEncoder needs 2D input — hence the double brackets
df['education_encoded'] = oe.fit_transform(df[['education']])

print(df[['education', 'education_encoded']].drop_duplicates().sort_values('education_encoded'))

        education  education_encoded
779     Preschool                0.0
323       1st-4th                1.0
37        5th-6th                2.0
9         7th-8th                3.0
54            9th                4.0
5            10th                5.0
0            11th                6.0
173          12th                7.0
1         HS-grad                8.0
3    Some-college                9.0
41      Assoc-voc               10.0
2      Assoc-acdm               11.0
11      Bachelors               12.0
15        Masters               13.0
7     Prof-school               14.0
19      Doctorate               15.0


## LabelEncoder vs OrdinalEncoder — Key Comparison

### LabelEncoder
- Works on **1D input** (single array)
- Assigns numbers **alphabetically** — no control over order
- Use only for **target variable `y`**

### OrdinalEncoder
- Works on **2D input** (one or more columns)
- You can define a **custom order** via `categories` parameter
- Use for **features `X`** that have a natural ranking

### Why LabelEncoder is wrong for features
LabelEncoder assigns numbers alphabetically — so `Preschool` gets 13 
and `10th` gets 0, which is completely wrong for education levels.
A model would interpret these numbers as meaningful ranks and learn 
incorrect relationships.

### The Rule
| Situation | Encoder |
|---|---|
| Target variable `y` | LabelEncoder |
| Feature with natural order | OrdinalEncoder with defined order |
| Feature with no natural order | OneHotEncoder |

In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['education_label_encoded'] = le.fit_transform(df['education'])

cols = ['education', 'education_encoded', 'education_label_encoded']
print(df[cols].drop_duplicates().sort_values('education_encoded'))

        education  education_encoded  education_label_encoded
779     Preschool                0.0                       13
323       1st-4th                1.0                        3
37        5th-6th                2.0                        4
9         7th-8th                3.0                        5
54            9th                4.0                        6
5            10th                5.0                        0
0            11th                6.0                        1
173          12th                7.0                        2
1         HS-grad                8.0                       11
3    Some-college                9.0                       15
41      Assoc-voc               10.0                        8
2      Assoc-acdm               11.0                        7
11      Bachelors               12.0                        9
15        Masters               13.0                       12
7     Prof-school               14.0                       14
19      

## OneHotEncoder

OneHotEncoder is used for **nominal categorical features** — categories with no natural order.
Instead of assigning a single number, it creates a **new binary column for each category**.

Use OneHotEncoder when:
- Categories have no meaningful ranking (e.g. workclass, occupation, race)
- The number of unique values is low to medium (typically < 15)

**Key rule:** Avoid OneHotEncoder for high cardinality columns (many unique values) 
as it creates too many new columns — use Target Encoding instead.

In [14]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Using workclass as example — 7 unique values, no natural order
encoded = ohe.fit_transform(df[['workclass']])

ohe_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(['workclass']))
print(df.head()['workclass'])
print()
print(ohe_df.head())

0      Private
1      Private
2    Local-gov
3      Private
5      Private
Name: workclass, dtype: str

   workclass_Federal-gov  workclass_Local-gov  workclass_Private  \
0                    0.0                  0.0                1.0   
1                    0.0                  0.0                1.0   
2                    0.0                  1.0                0.0   
3                    0.0                  0.0                1.0   
4                    0.0                  0.0                1.0   

   workclass_Self-emp-inc  workclass_Self-emp-not-inc  workclass_State-gov  \
0                     0.0                         0.0                  0.0   
1                     0.0                         0.0                  0.0   
2                     0.0                         0.0                  0.0   
3                     0.0                         0.0                  0.0   
4                     0.0                         0.0                  0.0   

   workclass_Witho

## Dummy Variables (pd.get_dummies)

`pd.get_dummies` is pandas' built-in alternative to OneHotEncoder.
It does the same thing — creates binary columns for each category.

### Key differences from OneHotEncoder
| | pd.get_dummies | OneHotEncoder |
|---|---|---|
| Library | pandas | sklearn |
| Input | DataFrame | 2D array |
| Use in Pipeline | No | Yes |
| drop_first | Yes | Yes (drop='first') |

### drop_first — what and why
With 7 workclass categories, you only need 6 binary columns — 
if all 6 are 0, the 7th is implied. The extra

In [15]:
# Basic get_dummies
dummies = pd.get_dummies(df['workclass'])
print(dummies.head())
print()

# With drop_first — removes first category to avoid multicollinearity
dummies_dropped = pd.get_dummies(df['workclass'], drop_first=True)
print(dummies_dropped.head())

   Federal-gov  Local-gov  Private  Self-emp-inc  Self-emp-not-inc  State-gov  \
0        False      False     True         False             False      False   
1        False      False     True         False             False      False   
2        False       True    False         False             False      False   
3        False      False     True         False             False      False   
5        False      False     True         False             False      False   

   Without-pay  
0        False  
1        False  
2        False  
3        False  
5        False  

   Local-gov  Private  Self-emp-inc  Self-emp-not-inc  State-gov  Without-pay
0      False     True         False             False      False        False
1      False     True         False             False      False        False
2       True    False         False             False      False        False
3      False     True         False             False      False        False
5      False     Tru

In [ ]:
# drop='first' — drops the first category per feature
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

In [ ]:
drop='if_binary'  # drops one category only for binary columns (e.g. gender)

## Target Encoding

Target Encoding replaces each category with the **mean of the target variable** 
for that category. It is specifically designed for **high cardinality categorical 
columns** where OneHotEncoder would create too many columns.

### How it works
For each unique category, compute:   
encoded_value = mean of target for all rows belonging to that category

### Target types
| Target Type | Problem | Output |
|---|---|---|
| `binary` | 2 classes (e.g. >50K / <=50K) | 1 column — proportion of positive class |
| `multiclass` | 3+ classes (e.g. low/medium/high) | k columns — one per class |
| `continuous` | Regression (e.g. actual salary) | 1 column — mean of target value |
| `auto` | sklearn detects automatically | depends |

### When to use Target Encoding
- High cardinality columns (typically > 15 unique values)
- When OneHotEncoder would create too many columns
- When OrdinalEncoder would introduce false ordering

### Warning — Data Leakage
Target encoding uses the target variable `y` to encode features.
If done incorrectly (fitting on full dataset before splitting), 
it leaks target information into features — always fit only on training data.

In [18]:
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import train_test_split

X = df.drop(columns=['income', 'education_encoded', 'education_label_encoded'])
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- Manual Target Encoding (from scratch) ----
y_train_binary = (y_train == '>50K').astype(int)

temp = X_train.copy()
temp['income'] = y_train_binary.values
country_means = temp.groupby('native-country')['income'].mean()

print("Manual Target Encoding (top 10):")
print(country_means.sort_values(ascending=False).head(10))
print()

# ---- sklearn TargetEncoder — binary ----
te_binary = TargetEncoder(target_type='binary')
X_train_binary = te_binary.fit_transform(X_train[['native-country']], y_train)
X_test_binary = te_binary.transform(X_test[['native-country']])
print("sklearn TargetEncoder (binary) — first 5 rows:")
print(X_train_binary[:5])
print()

# ---- sklearn TargetEncoder — auto ----
te_auto = TargetEncoder(target_type='auto')
X_train_auto = te_auto.fit_transform(X_train[['native-country']], y_train)
print("sklearn TargetEncoder (auto) — first 5 rows:")
print(X_train_auto[:5])

Manual Target Encoding (top 10):
native-country
Taiwan      0.487179
India       0.410256
England     0.392157
Greece      0.390244
Cambodia    0.388889
France      0.387097
Italy       0.360000
Canada      0.350365
Iran        0.333333
Japan       0.317460
Name: income, dtype: float64

sklearn TargetEncoder (binary) — first 5 rows:
[[0.25163515]
 [0.25131486]
 [0.25131486]
 [0.25163515]
 [0.25224557]]

sklearn TargetEncoder (auto) — first 5 rows:
[[0.25185193]
 [0.25201203]
 [0.25201203]
 [0.251077  ]
 [0.25177633]]


## Smoothing in Target Encoding

### The Problem with Raw Target Encoding
Rare categories have unreliable means — if only 1 person is from 
Holand-Netherlands and they earn <=50K, the raw mean is 0.0. 
This is not representative and causes the model to overfit on rare categories.

### The Solution — Smoothing
Smoothing blends the category mean with the global mean:
smoothed = (n × category_mean + smooth × global_mean) / (n + smooth)

Where:
- `n` = number of samples in that category
- `smooth` = smoothing factor (higher = more pull toward global mean)
- `global_mean` = overall target mean across all training data

### Effect of Smoothing
| smooth | Rare category | Frequent category |
|---|---|---|
| 0 | Raw mean (unreliable) | Raw mean |
| 1 | Slight pull toward global mean | Almost no change |
| 10 | Strong pull toward global mean | Small change |
| 100 | Almost equal to global mean | Moderate change |

### Key Insight
- **Rare categories** get pulled strongly toward the global mean
- **Frequent categories** resist the pull — enough samples to be reliable
- Default `smooth='auto'` in sklearn lets it compute the optimal value

### When to increase smoothing
- Dataset has many rare categories
- High cardinality columns with uneven distribution
- Model is overfitting on rare category values

In [19]:
# Compare different smoothing strengths
for smooth in [0, 1, 10, 100]:
    te = TargetEncoder(target_type='binary', smooth=smooth)
    encoded = te.fit_transform(X_train[['native-country']], y_train)
    
    # Check a rare country — Holand-Netherlands
    sample = pd.DataFrame({'native-country': ['Holand-Netherlands', 'India', 'United-States']})
    result = te.transform(sample)
    print(f"smooth={smooth}: Holand={result[0][0]:.4f}, India={result[1][0]:.4f}, US={result[2][0]:.4f}")

smooth=0: Holand=0.0000, India=0.4103, US=0.2517
smooth=1: Holand=0.1232, India=0.4089, US=0.2517
smooth=10: Holand=0.2241, India=0.3974, US=0.2517
smooth=100: Holand=0.2440, India=0.3348, US=0.2517


## Summary — Encoding Techniques

| Encoder | Input | Use Case | Handles Order |
|---|---|---|---|
| LabelEncoder | 1D (target only) | Binary/multi-class target `y` | No — alphabetical |
| OrdinalEncoder | 2D (features) | Features with natural ranking | Yes — custom order |
| OneHotEncoder | 2D (features) | Nominal features, low cardinality | No order needed |
| pd.get_dummies | DataFrame | Quick EDA, not for pipelines | No order needed |
| TargetEncoder | 2D (features) | High cardinality features |